# Unit 6 Project
### Creating Designs by Leveraging Stable Diffusion and Gradio UI


In [ ]:
%pip install --upgrade pip
%pip uninstall -y torch torchvision triton-windows diffusers gradio

%pip install  gradio 
%pip install diffusers[torch]
%pip install triton-windows
%pip install torchvision  
%pip install  torch  --index-url https://download.pytorch.org/whl/cu130
%pip install transformers



In [ ]:
# imports
import gc
import gradio as gr
import torch
import torchvision
import transformers
print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
from diffusers import DiffusionPipeline, StableDiffusion3Pipeline


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()

def generate_image (prompt, negative_prompt, guidance_scale, seed):
    # Take model and generate an image from prompt and seed
    # model_id = "stabilityai/stable-diffusion-xl-base-1.0"
    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, use_safetensors=True, variant="fp16", safety_checker=None)
    pipe = pipe.to(device)
    #pipe.safety_checker = lambda images, **kwargs: (images, [False] * len(images))
    # refiner = DiffusionPipeline.from_pretrained(
    # "stabilityai/stable-diffusion-xl-refiner-1.0",
    #     text_encoder_2=pipe.text_encoder_2,
    #     vae=pipe.vae,
    #     torch_dtype=torch.float16,
    #     use_safetensors=True,
    #     variant="fp16",
    #     )
    # refiner = refiner.to(device)
    generator = torch.Generator(device).manual_seed(seed)
    pipe.to(device)
    pipe.enable_model_cpu_offload()
    image = pipe(prompt, negative_prompt=negative_prompt, generator=generator, height=328, width=720, guidance_scale=guidance_scale).images
    # image1 = refiner(prompt, image=image).images[0]
    gc.collect()
    torch.cuda.empty_cache()
    
    # If torch.compile was used:
    torch._dynamo.reset()
    torch.cuda.synchronize()
    return image[0]

In [ ]:
def_prompt = "Create a banner ad for 'Gears Ltd.' They make gears. Use blue for background color, and steel for the gears"
demo = gr.Interface(fn=generate_image , inputs=[gr.Textbox(label="Prompt", value=def_prompt),
                                             gr.Textbox(label="Negative Prompt", value="text, words, logo, watermarks, signatures"),
                                             gr.Slider(0, 20, step=0.1, label="Guidance Scale", value=7.5),
                                             gr.Number(label="Seed", value=691)],
                                     outputs=[gr.Image(label="Banner Background")],)

demo.launch()